In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("social_media_productivity_6000.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    5880 non-null   float64
 1   daily_screen_time      5880 non-null   float64
 2   social_media_hours     5880 non-null   float64
 3   study_hours            5880 non-null   float64
 4   sleep_hours            5880 non-null   float64
 5   notifications_per_day  5880 non-null   float64
 6   focus_score            5880 non-null   float64
 7   addiction_level        5880 non-null   object 
 8   productivity_score     5880 non-null   float64
dtypes: float64(8), object(1)
memory usage: 422.0+ KB


In [4]:
df.sample(10)

,age,daily_screen_time,social_media_hours,study_hours,sleep_hours,notifications_per_day,focus_score,addiction_level,productivity_score
1918,23.0,11.99,6.69,1.94,4.79,273.0,78.90,High,0.00
1245,23.0,8.31,6.86,3.19,4.38,70.0,82.90,High,18.38
2054,34.0,4.45,2.13,2.94,6.60,161.0,100.00,Medium,25.36
4038,38.0,5.95,4.84,6.96,6.50,190.0,100.00,Medium,59.72
4568,22.0,8.41,5.83,1.33,7.00,184.0,84.82,High,0.00
2585,24.0,3.91,1.52,1.48,4.15,154.0,100.00,Low,10.89
732,22.0,11.09,7.73,5.47,5.39,153.0,91.20,High,NaN
4841,23.0,4.89,1.62,2.04,8.69,66.0,100.00,Low,46.40
2157,26.0,3.29,NaN,6.17,4.57,187.0,100.00,Medium,50.23
4900,39.0,4.55,3.71,1.16,5.66,275.0,90.37,Medium,0.00


In [5]:
df["addiction_level"].value_counts()

addiction_level
Medium    3064
High      1857
Low        959
Name: count, dtype: int64

## **Handling missing values**

In [6]:
df.isnull().any()

age                      True
daily_screen_time        True
social_media_hours       True
study_hours              True
sleep_hours              True
notifications_per_day    True
focus_score              True
addiction_level          True
productivity_score       True
dtype: bool

In [7]:
def missing_values(df):
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == "object":
                df[col].fillna(df[col].mode()[0], inplace=True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)
    return df


In [8]:
df = missing_values(df)

## **Encoding**
There is only one column that has "object" values and we need to encode this column.

In [9]:
addiction_level_mapping = {
    "Low" : 0,
    "Medium" : 1,
    "High" : 2
}

df["addiction_level"] = df["addiction_level"].map(addiction_level_mapping)

## **Scaling**

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
def scaling(df):
    scaler = MinMaxScaler()
    num_col = df.select_dtypes(include=['int64','float64', 'int32']).columns.drop("addiction_level")
    df[num_col] = scaler.fit_transform(df[num_col])
    return df
    

In [12]:
df = scaling(df)

In [13]:
df.sample(10)

,age,daily_screen_time,social_media_hours,study_hours,sleep_hours,notifications_per_day,focus_score,addiction_level,productivity_score
4197,0.875000,0.333333,0.198,0.64000,0.230,0.218638,1.000000,1,0.5379
663,1.000000,0.306306,0.102,0.16125,0.610,0.146953,1.000000,0,0.3385
4315,0.833333,0.149149,0.129,0.23375,0.272,0.365591,1.000000,0,0.1508
2346,0.500000,0.153153,0.144,0.61375,0.606,0.820789,1.000000,1,0.6768
1331,0.291667,0.983984,0.565,0.72375,0.478,0.763441,0.964646,2,0.2395
783,0.916667,0.849850,0.607,0.94125,0.854,0.455197,1.000000,2,0.7775
5272,0.250000,0.962963,0.779,0.14125,0.454,0.179211,0.620604,2,0.0000
188,0.250000,0.890891,0.606,0.43500,0.428,0.229391,0.934233,2,0.1174
2420,0.125000,0.143143,0.125,0.92125,0.798,0.103943,1.000000,0,1.0000
1216,0.291667,0.769770,0.751,0.79500,0.358,0.745520,0.997149,2,0.3273


## **Logistic Regression**

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

x = df.drop("addiction_level", axis=1)
y = df["addiction_level"]

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)
class_rep = classification_report(y_test, y_pred)
print(class_rep)


              precision    recall  f1-score   support

           0       0.87      0.78      0.82       199
           1       0.89      0.93      0.91       651
           2       0.94      0.91      0.92       350

    accuracy                           0.90      1200
   macro avg       0.90      0.87      0.89      1200
weighted avg       0.90      0.90      0.90      1200



## **Decision Tree Classifier**

In [15]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()

dtc.fit(x_train, y_train)

y_pred = dtc.predict(x_test)

cl_report = classification_report(y_test, y_pred)
print(cl_report)

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       199
           1       0.98      0.96      0.97       651
           2       0.95      0.97      0.96       350

    accuracy                           0.97      1200
   macro avg       0.97      0.97      0.97      1200
weighted avg       0.97      0.97      0.97      1200



## **Random Forest Classifier**

In [16]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()

rfc.fit(x_train, y_train)

y_pred = rfc.predict(x_test)

clrp = classification_report(y_test, y_pred)
print(clrp)

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       199
           1       0.99      0.97      0.98       651
           2       0.96      0.98      0.97       350

    accuracy                           0.98      1200
   macro avg       0.98      0.98      0.98      1200
weighted avg       0.98      0.98      0.98      1200



## **Support Vector Classifier**

In [17]:
from sklearn.svm import SVC

svc = SVC(kernel="linear", C=1.0)

svc.fit(x_train, y_train)

y_pred = svc.predict(x_test)

classr = classification_report(y_test, y_pred)
print(classr)


              precision    recall  f1-score   support

           0       0.91      0.87      0.89       199
           1       0.93      0.94      0.93       651
           2       0.94      0.93      0.93       350

    accuracy                           0.93      1200
   macro avg       0.92      0.92      0.92      1200
weighted avg       0.93      0.93      0.93      1200



## **Comaparison**

In [18]:
from tabulate import tabulate

columns = ["Model", "Accuracy", "Precision", "Recall", "F1_score"]

result = [["Logistic Regression", 0.90, 0.87, 0.78, 0.82],
          ["Decision Tree Classifier", 0.97, 0.97, 0.97, 0.97],
          ["Random Forest Classifier", 0.98, 0.98, 0.99, 0.98],
          ["Support Vector Classifier", 0.93, 0.91, 0.87, 0.89]
          ]

table = tabulate(result, headers = columns, tablefmt="grid", floatfmt='.2f')
print(table)

+---------------------------+------------+-------------+----------+------------+
| Model                     |   Accuracy |   Precision |   Recall |   F1_score |
+===========================+============+=============+==========+============+
| Logistic Regression       |       0.90 |        0.87 |     0.78 |       0.82 |
+---------------------------+------------+-------------+----------+------------+
| Decision Tree Classifier  |       0.97 |        0.97 |     0.97 |       0.97 |
+---------------------------+------------+-------------+----------+------------+
| Random Forest Classifier  |       0.98 |        0.98 |     0.99 |       0.98 |
+---------------------------+------------+-------------+----------+------------+
| Support Vector Classifier |       0.93 |        0.91 |     0.87 |       0.89 |
+---------------------------+------------+-------------+----------+------------+


# **Regression Algorithms**

In [19]:
df.sample(10)

,age,daily_screen_time,social_media_hours,study_hours,sleep_hours,notifications_per_day,focus_score,addiction_level,productivity_score
2152,0.504641,0.856857,0.792,0.715000,0.874,0.641577,0.860863,2,0.2637
4274,0.375000,0.951952,0.632,0.652500,0.282,0.017921,1.000000,2,0.0205
1838,0.416667,0.657658,0.384,0.880000,0.086,0.186380,0.930879,1,0.4714
539,0.375000,0.555556,0.230,0.900000,0.526,0.698925,1.000000,1,0.6080
263,0.000000,0.634635,0.415,0.057500,0.376,0.014337,0.760502,1,0.0000
5671,0.375000,0.346346,0.196,0.488750,0.720,0.594982,1.000000,1,0.4368
3849,0.250000,0.986987,0.350,0.508666,0.788,0.207885,1.000000,1,0.2334
1787,0.125000,0.859860,0.886,0.508666,0.378,0.731183,0.930879,2,0.3312
5023,0.708333,0.494494,0.471,0.917500,0.630,0.272401,1.000000,2,0.4912
5211,0.375000,0.470470,0.324,0.791250,0.878,0.444444,1.000000,1,0.6295


In [20]:
x = df.drop("productivity_score", axis=1)
y= df["productivity_score"]

In [21]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

## **LInear Regression**

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr = LinearRegression()

lr.fit(x_train, y_train)

y_pred = lr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)

0.08506650639554562
0.01239830883386489
0.11134769343756022
0.8317685760231144


## **Decision Tree Regressor**


In [23]:
from sklearn.tree import DecisionTreeRegressor

dtr = DecisionTreeRegressor()

dtr.fit(x_train, y_train)

y_pred = dtr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)


0.11091606539115645
0.023253518837452802
0.15249104510577924
0.6844753071634408


## **Random Forest Regressor**

In [24]:
from sklearn.ensemble import RandomForestRegressor

rfr = RandomForestRegressor()

rfr.fit(x_train, y_train)

y_pred = rfr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)


0.08212830197619049
0.012024297566278248
0.10965535812844827
0.836843497850972


## **Support Vector Regressor**

In [25]:
from sklearn.svm import SVR

svr = SVR()

svr.fit(x_train, y_train)

y_pred = svr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)


0.08186887443595872
0.011386507696261884
0.10670758031303064
0.845497605396496


## **Comparison**

In [26]:
columns = ["Algorithm", "Mean Absolute Error", "Mean Squared Error", "Root Mean Squared Error", "R2_score"]

results = [["Linear Regression", 0.085, 0.012, 0.111, 0.831],
           ["Decision Tree Regressor", 0.111, 0.023, 0.152, 0.68],
           ["Random Forest Regressor", 0.082, 0.012, 0.109, 0.836],
           ["Support Vector Regressor", 0.81, 0.011, 0.106, 0.845]]

table = tabulate(results, headers = columns, tablefmt = 'grid', floatfmt='.2f')
print(table)

+--------------------------+-----------------------+----------------------+---------------------------+------------+
| Algorithm                |   Mean Absolute Error |   Mean Squared Error |   Root Mean Squared Error |   R2_score |
+==========================+=======================+======================+===========================+============+
| Linear Regression        |                  0.09 |                 0.01 |                      0.11 |       0.83 |
+--------------------------+-----------------------+----------------------+---------------------------+------------+
| Decision Tree Regressor  |                  0.11 |                 0.02 |                      0.15 |       0.68 |
+--------------------------+-----------------------+----------------------+---------------------------+------------+
| Random Forest Regressor  |                  0.08 |                 0.01 |                      0.11 |       0.84 |
+--------------------------+-----------------------+------------